# Notebook for show-casing different setups for running the HBV-SASK model using UQEF-Dynamic

## This notebook is meant just to run (multiple) models runs without the statistics/processing part

# Import

In [ ]:
import numpy as np
import pathlib
import pandas as pd
import sys
import time

In [ ]:
# TODO - change this path accordingly
# sys.path.insert(1, '/work/ga45met/Hydro_Models/HBV-SASK-py-tool')
sys.path.insert(1, '/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic')

In [ ]:
from uqef_dynamic.utils import utility
from uqef_dynamic.models.hbv_sask import hbvsask_utility as hbv
from uqef_dynamic.models.hbv_sask import HBVSASKModel as hbvmodel

In [ ]:
# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

import matplotlib.pyplot as mp

from plotly.offline import plot

pd.options.plotting.backend = "plotly"

# Defining paths

In [ ]:
# TODO - change these paths accordingly
hbv_model_data_path = pathlib.Path("/work/ga45met/Hydro_Models/HBV-SASK-data")
configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_6D.json')
# configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_10D_full.json')
# configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_10D_MC_banff.json')
# configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Hydro/configurations/configuration_hbv_6D.json')
basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'

inputModelDir = hbv_model_data_path

# TODO - change this path accordingly
workingDir = hbv_model_data_path / basis / "model_runs" / 'ensamble_run_full' #"whole_time_generating_state_df"


# Experimenting with different QoIs

# Experiment 1 - Sliding Window GoF

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":"GoF",
    "qoi_column":"Q_cms",
    "autoregressive_model_first_order":"False",
    "transform_model_output":"None",
    "read_measured_data": "True",
    "qoi_column_measured":"streamflow",
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"sliding_window",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"False",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "False",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")


In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

# Experiment 2 - Multiple QoI with transformation

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":["Q_cms","AET"],
    "qoi_column":["Q_cms","AET"],
    "autoregressive_model_first_order":"False",
    "transform_model_output":["log", "None"],
    "read_measured_data": ["True","False"],
    "qoi_column_measured":["streamflow","None"],
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"continuous",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"False",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "False",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
hbvsaskModelObject.time_series_measured_data_df

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

# Experiment 3 - gradient computation

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":["Q_cms","AET"],
    "qoi_column":["Q_cms","AET"],
    "autoregressive_model_first_order":"False",
    "transform_model_output":["None", "None"],
    "read_measured_data": ["True","False"],
    "qoi_column_measured":["streamflow","None"],
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"continuous",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"True",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "True",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(
    createNewFolder=createNewFolder,
)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
len(results_array)

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
results_array[0][0]['parameters_dict']

## Experiment 3.2 - gradient computation - multiple parameters list

In [ ]:
start = time.time()
list_of_parmeter_beta_values = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
print(f"list_of_parmeter_beta_values - {type(list_of_parmeter_beta_values)}")
results_array = hbvsaskModelObject.run(
    i_s = range(0,len(list_of_parmeter_beta_values)),
    parameters = list_of_parmeter_beta_values,
    createNewFolder=createNewFolder,
    merge_output_with_measured_data=True
)
end = time.time()
runtime = end - start
print(f"#{len(list_of_parmeter_beta_values)}  execution(s) of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
len(results_array)

In [ ]:
type(results_array[0])

In [ ]:
type(results_array[0][0])

In [ ]:
results_array[0][0].keys()

In [ ]:
results_array[1][0]['result_time_series']

In [ ]:
results_array[1][0]['parameters_dict']['beta']

In [ ]:
fig = go.Figure()
for indx in range(0, len(results_array)):
    temp = results_array[indx][0]['result_time_series']
    fig.add_trace(go.Scatter(x=temp.index,y=temp["d_Q_cms_d_beta"], name=results_array[indx][0]['parameters_dict']['beta'],))
fig.show()

In [ ]:
fig = go.Figure()
for indx in range(0, len(results_array)):
    temp = results_array[indx][0]['result_time_series']
    fig.add_trace(go.Scatter(x=temp.index,y=temp["d_AET_d_beta"], name=results_array[indx][0]['parameters_dict']['beta'],))
fig.show()

In [ ]:
results_array[4][0]['parameters_dict']

In [ ]:
hbvsaskModelObject.compute_active_subspaces

In [ ]:
results_array[0][0]['grad_matrix'].keys()

In [ ]:
type(results_array[0][0]['grad_matrix']['Q_cms'])

In [ ]:
print(len(results_array[0][0]['grad_matrix']['Q_cms']))
simulation_range = pd.date_range(
    start=hbvsaskModelObject.start_date_predictions, end=hbvsaskModelObject.end_date, freq="1D")
print(len(simulation_range))

In [ ]:
len(results_array[0][0]['grad_matrix']['Q_cms'])

# Experiment 4 - Autoregressive Model

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff_autoregressive'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":["Q_cms","AET"],
    "qoi_column":["Q_cms","AET"],
    "autoregressive_model_first_order":"True",
    "transform_model_output":["log", "None"],
    "read_measured_data": ["True","False"],
    "qoi_column_measured":["streamflow","None"],
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"continuous",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"False",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "False",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
# now, columns of interest are - delta_Q_cms and delta_AET
results_array[0][0]['result_time_series']